# E5 · 애매한 문항만 더 큰 해상도로 다시 채점 (학습 없음)

**왜**: 대회 이미지는 대부분 720×960(약 0.7MP)입니다. 지금 쓰는 up2.0(2.7~3.4MP)도 이미 가로세로 약 2배로 키운 것이고, native보다 up2.0 점수가 높았습니다. 원본에 없는 글씨가 새로 생기지는 않지만, 크게 키우면 작은 글자 하나가 더 많은 이미지 토큰에 걸쳐 들어가 모델이 구분하기 쉬워질 수 있습니다. 그래서 **up3.0(약 3배), up4.0(약 4배)**까지 올려 봅니다.

**하는 일**
- E9와 같은 애매한 문항(세 앙상블 중 하나라도 1·2위 확률 차 < 0.8)만 **Q35·Q38**을 기존 어댑터 그대로 up3.0·up4.0으로 다시 채점합니다. Gemma는 이미지 토큰 1,120개가 최대라 더 못 올리고, HCX는 설치가 달라 이 노트북에서 빠집니다(원래 확률 그대로).
- 먼저 검증 문항을 up2.0으로 다시 채점해 기존 예측이 재현되는지 확인합니다(85% 이상이어야 정상). 해상도마다 letter_mass와 이미지 토큰 수도 기록합니다.
- 해상도를 올려 앙상블 답이 바뀐 검증 문항은 `e5_changed_val.html`에 이미지와 함께 저장합니다. 작은 글씨를 정말 더 잘 읽었는지 눈으로 볼 수 있습니다.

**사전 기준 (결과를 보기 전에 고정)**
- **H1**: 애매한 문항에서 Q35·Q38 확률을 {up2.0(기존), up3.0, up4.0} 동일 가중 평균으로 바꿉니다. 메모리 부족으로 빠진 해상도는 평균에서 제외합니다. 해상도 하나만 쓴 결과는 참고용입니다.
- 검증 순이득 + clean4 후보 공개 순이득 ≥ +3이고 공개 순이득 ≥ 0 → 채택. 검증 순이득 ≤ −3이면 공개 확인 없이 불채택. 검증에서 바뀐 답이 10개 미만이면 검증만으로 판단하지 않습니다.

**실행**: 위에서부터 끝까지. 끊기면 다시 실행하면 남은 채점만 합니다. up4.0은 문항당 이미지 토큰이 약 1만 개라 배치 1로 느리게 돕니다. 메모리가 모자라면 그 해상도만 건너뛰고 기록합니다.
**런타임**: G4 권장 (A100 80GB도 가능). 결과는 `vqa_runs/e5_hires/`에만 씁니다.

## 1. 설정

In [ ]:
MEMBERS_HR = ["q35_27b_r16_native", "q38_27b_r16_native"]
#   고해상도로 다시 채점할 멤버. Qwen(픽셀 기준 해상도)만 가능 — Gemma는 이미지 토큰 1,120개가 최대라 더 못 올림, HCX는 설치가 달라 이 노트북으로 안 됨
HR_PRESETS = ["up3.0", "up4.0"]   # 더 키울 해상도 (원본 720×960 기준 가로세로 약 3배·4배). 메모리가 모자라면 그 해상도만 자동으로 건너뜀
HR_BS      = {"up2.0": 8, "up3.0": 2, "up4.0": 1}   # 해상도별 채점 배치 (OOM이면 자동으로 절반)
PRESET     = "up2.0"        # 기존 멤버 예측 파일의 해상도 (기준)
TAU_GATE   = 0.8            # 기준 앙상블 3개 중 하나라도 1·2위 확률 차 < 0.8인 문항만 채점 (E9와 같은 문항)
RUN_TEST   = True           # test 애매한 문항도 채점 (제출 후보를 만들려면 필요)
RUN_NAME   = "e5_hires"     # 결과 폴더: vqa_runs/e5_hires/<멤버>/  (기존 멤버 폴더는 읽기만 함)
MODEL_ID   = "Qwen/Qwen3.5-27B"   # 자리표시(2번 셀 설치 조건용). 멤버별 베이스 모델은 어댑터 기록에서 자동으로 읽음
DELETE_HF_CACHE = True      # 다른 베이스로 넘어갈 때 이전 베이스 파일을 지워 디스크 확보
RETRY_SKIPPED = False       # True: 예전 실행에서 메모리 부족으로 건너뛴 해상도를 다시 시도 (더 큰 GPU로 바꿨을 때)

# ── 기준 앙상블 레시피 (E8·E9와 같음) ──
Q35, Q38 = ("q35_27b_r16_native", PRESET), ("q38_27b_r16_native", PRESET)
GDEV, G, HCX = ("gemma4_31b_r16_up20_dev", PRESET), ("gemma4_31b_r16_up20", PRESET), ("hcx32b_r16_native", PRESET)
RECIPES = {
    "clean4_qg_hcx":    dict(file="submission_clean4_qg_hcx.csv",    members=[Q35, Q38, GDEV, G, HCX], weights=[1, 1, 1, 1, 2], cal=False),
    "ens_q38_gemma4":   dict(file="submission_ens_q38_gemma4.csv",   members=[Q38, G],                 weights=[1, 1],          cal=False),
    "ens_q38_gdev_cal": dict(file="submission_ens_q38_gdev_cal.csv", members=[Q38, GDEV],              weights=[1, 1],          cal=True),
}
VOTE = dict(name="majority_vote", file="submission_majority_vote.csv", voters=["ens_q38_gemma4", "clean4_qg_hcx", "ens_q38_gdev_cal"])
PUBLIC = {"clean4_qg_hcx": 0.97646, "majority_vote": 0.97676}; N_PUBLIC = 3357
ENSEMBLE_SUBDIR = "ensembles"

# ── 사전 기준 (결과를 보기 전에 고정) ──
# H1: 애매한 문항에서 Q35·Q38 확률을 {up2.0(기존), up3.0, up4.0} 동일 가중 평균으로 바꿈 (메모리 부족으로 빠진 해상도는 제외)
NET_MIN = 3        # 검증 순이득 + 공개 순이득(clean4 후보) ≥ +3 이고 공개 순이득 ≥ 0 → 채택 / 검증 순이득 ≤ −3이면 공개 확인 없이 불채택

DRIVE_ZIP    = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "YOUR_ZIP_PASSWORD"
DATA_DIR     = "/content"
OUT_DIR      = "/content/drive/MyDrive/vqa_runs"
BATCH_SIZE   = "auto"
SEED, VAL_SIZE = 42, 1000
SKIP_MISSING_IMAGES = False
HF_TOKEN     = ""

RES_PRESETS = {
    "baseline384": (384 * 384, 384 * 384),
    "mid":         (256 * 28 * 28, 512 * 28 * 28),
    "native":      (256 * 28 * 28, 1280 * 28 * 28),
    "up1.5":       (1_550_000, 2_100_000),
    "up2.0":       (2_700_000, 3_400_000),
    "up3.0":       (6_000_000, 7_600_000),      # 720×960 사진 → 약 2,120×2,830 (가로세로 약 2.95배)
    "up4.0":       (10_500_000, 12_500_000),    # 720×960 사진 → 약 2,810×3,740 (가로세로 약 3.9배)
}
RES_PRESETS_GEMMA4 = {"baseline384": 70, "mid": 140, "native": 280, "up1.5": 560, "up2.0": 1120}
RES_PRESET = PRESET
import os; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

## 2. 라이브러리 설치 (학습 때와 같은 버전)

In [ ]:
import re
!pip -q install "transformers==5.17.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "peft==0.21.0"
# Colab에 기본 설치된 구버전 torchao가 있으면 peft 0.21이 LoRA를 붙이는 순간 ImportError를 냅니다. 이 프로젝트에서는 쓰지 않으므로 제거
!pip -q uninstall -y torchao
# Qwen3.5의 선형 어텐션용 고속 Triton 커널. 없으면 순수 PyTorch 대체 구현으로 돌아 학습 시 메모리 부족(OOM)이 나고 매우 느립니다.
# (transformers가 import 시점에 이 패키지를 찾으므로, 이미 모델을 로드한 세션에서 설치했다면 '세션 다시 시작'이 필요합니다)
if re.search(r"Qwen3\.[5-9]", MODEL_ID):      # Qwen3.5 / 3.6 / 3.8 — 모두 같은 선형 어텐션 구조(model_type qwen3_5)
    !pip -q install "flash-linear-attention==0.5.2"

## 3. Drive 연결 → 멤버 확인 → 데이터 준비

In [ ]:
import os, re, json, hashlib, subprocess, shutil
try:
    from google.colab import drive
    drive.mount("/content/drive"); IN_COLAB = True
except ImportError:
    IN_COLAB = False
# ── 멤버 확인 (읽기만 함): 어댑터·베이스 모델·기존 예측 파일. 이름이 틀렸으면 압축을 풀기 전에 멈춤 ──
MEMBER_INFO = {}
for run in MEMBERS_HR:
    d = os.path.join(OUT_DIR, run); assert os.path.isdir(d), f"멤버 폴더가 없습니다: {d} → 1번 셀 MEMBERS_HR를 확인하세요"
    fs = sorted(os.listdir(d))
    lg = json.load(open(os.path.join(d, "train_log.json"), encoding="utf-8")) if os.path.exists(os.path.join(d, "train_log.json")) else {}
    cfg = lg.get("config") or {}
    assert not cfg.get("TRAIN_ON_FULL"), f"{run}은 전체 데이터로 학습한 실행입니다 → 이 노트북은 검증셋을 뺀 멤버만 씁니다"
    rv = {int(m.group(1)): f for f in fs if (m := re.match(rf"^preds_final_ep(\d+)__{re.escape(PRESET)}__1000\.csv$", f))}
    ea = {int(k): v for k, v in (lg.get("epoch_acc") or {}).items()}
    ep = next(iter(rv)) if len(rv) == 1 else (max(ea, key=lambda e: (ea[e], e)) if ea else (max(rv) if rv else None))
    assert ep in rv, f"{run}: {PRESET} 검증 예측 파일이 없습니다 (폴더 내용 일부: {fs[:8]})"
    rt = next((f for f in fs if re.match(rf"^preds_final_ep{ep}_test__{re.escape(PRESET)}__6714\.csv$", f)), None)
    ad = os.path.join(d, f"adapter_ep{ep}")
    assert os.path.isfile(os.path.join(ad, "adapter_model.safetensors")), f"{run}: 어댑터가 없습니다: {ad}"
    acfg = json.load(open(os.path.join(ad, "adapter_config.json"), encoding="utf-8"))
    mid = str(acfg.get("base_model_name_or_path") or "")
    if not mid or mid.startswith("/"): mid = str(cfg.get("MODEL_ID") or "")
    assert mid and not mid.startswith("/"), f"{run}: 베이스 모델 이름을 찾지 못했습니다"
    assert not ("hyperclova" in mid.lower() or "hcx" in run.lower()), f"{run}: HCX는 이 노트북으로 채점할 수 없습니다 → MEMBERS_HR에서 빼세요"
    assert "gemma" not in mid.lower() and "qwen" in mid.lower(), f"{run}({mid}): 픽셀 기준 해상도를 쓰는 Qwen 멤버만 됩니다 (Gemma는 이미지 토큰 1,120개가 최대) → MEMBERS_HR에서 빼세요"
    MEMBER_INFO[run] = dict(dir=d, ep=ep, adapter_dir=ad, model_id=mid, ref_val=os.path.join(d, rv[ep]), ref_test=os.path.join(d, rt) if rt else None,
                            md5=hashlib.md5(open(os.path.join(ad, "adapter_model.safetensors"), "rb").read(1 << 20)).hexdigest()[:8])
    print(f"  {run}: 베이스 {mid} | adapter_ep{ep} | 검증 {rv[ep]} | test {rt or '없음'}")
    if RUN_TEST: assert rt, f"{run}: test 예측 파일이 없습니다 (RUN_TEST=True)"
if IN_COLAB and not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    print("압축 해제 중... (몇 분 걸립니다)")
    r = subprocess.run(["7z", "x", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_DIR}", DRIVE_ZIP], capture_output=True, text=True)
    print(r.stdout[-400:]); assert r.returncode == 0, r.stderr[-800:]
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception: pass
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False); print("Hugging Face 로그인 완료")
for f in ["train.csv", "test.csv", "sample_submission.csv"]:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"{f} 가 {DATA_DIR} 에 없습니다"
E5_DIR = os.path.join(OUT_DIR, RUN_NAME); os.makedirs(E5_DIR, exist_ok=True)
INFO_PATH = os.path.join(E5_DIR, "e5_info.json")
E5_INFO = json.load(open(INFO_PATH, encoding="utf-8")) if os.path.exists(INFO_PATH) else {}
for run, inf in MEMBER_INFO.items():
    old = (E5_INFO.get("members") or {}).get(run)
    if old and old.get("md5") != inf["md5"]:
        raise RuntimeError(f"{E5_DIR}/{run}에는 다른 어댑터로 만든 결과가 있습니다 → 1번 셀 RUN_NAME을 바꾸세요")
print("데이터 OK:", DATA_DIR, "| 결과 폴더(새로 씀):", E5_DIR)

## 4. 공통 import · CSV 로드

In [ ]:
import re, json, time, math, hashlib, random, datetime, platform
import numpy as np, pandas as pd, torch
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def _fix_nvrtc():
    # 일부 Colab 이미지(새 CUDA)에서 GPU 정수 텐서의 prod가 런타임 컴파일(NVRTC) 도중 내장 라이브러리를 못 찾아 실패함.
    # 1) 그 라이브러리를 경로로 직접 불러오고, 2) 그래도 안 되면 정수 prod만 CPU에서 계산하도록 우회 (결과는 완전히 같음)
    if not torch.cuda.is_available(): return
    import glob, ctypes
    def _ok():
        try: torch.tensor([[1, 2, 3]], device="cuda").prod(-1).item(); return True, ""
        except RuntimeError as e: return False, str(e)
    good, msg = _ok()
    if good: return
    need = re.findall(r"libnvrtc-builtins\.so[\d.]*", msg)
    pats = ["/usr/local/lib/python3*/dist-packages/nvidia/**/libnvrtc-builtins.so*", "/usr/local/lib/python3*/site-packages/nvidia/**/libnvrtc-builtins.so*",
            "/usr/local/cuda*/lib64/libnvrtc-builtins.so*", "/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so*"]
    found = sorted({f for p in pats for f in glob.glob(p, recursive=True)})
    for f in found:
        if not need or any(os.path.basename(f).startswith(n) for n in need):
            try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except OSError: pass
    if _ok()[0]: print("✅ NVRTC 내장 라이브러리를 직접 불러와 해결:", need); return
    _orig_prod = torch.Tensor.prod
    def _prod(self, *a, **k):
        if self.is_cuda and not (self.is_floating_point() or self.is_complex()):
            return _orig_prod(self.cpu(), *a, **k).to(self.device)
        return _orig_prod(self, *a, **k)
    torch.Tensor.prod = _prod
    print("⚠️ GPU 정수 prod를 CPU에서 계산하도록 우회 (결과 동일) | 확인:", _ok()[0], "| 찾은 라이브러리:", found or "없음")
_fix_nvrtc()

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_df   = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_DIR, "dev.csv")) if os.path.exists(os.path.join(DATA_DIR, "dev.csv")) else None
for c in ["question", "a", "b", "c", "d"]:
    train_df[c] = train_df[c].astype(str); test_df[c] = test_df[c].astype(str)
train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
assert set(train_df.answer.unique()) <= set("abcd"), train_df.answer.unique()
assert list(test_df.id) == list(sub_df.id), "test.csv 와 sample_submission.csv 의 id 순서가 다릅니다"
print(f"train {len(train_df):,} | test {len(test_df):,} | dev {0 if dev_df is None else len(dev_df):,}")
print("정답 분포:", train_df.answer.value_counts().sort_index().to_dict())

def img_path(rel):            # CSV의 path 컬럼은 DATA_DIR 기준 상대경로
    return os.path.join(DATA_DIR, rel)

## 5. 고정 검증셋 분할

In [ ]:
EXPECTED_SPLIT_MD5 = "df05e4bc19d27ebceeef2570672149c2"

def make_split(df, val_size=VAL_SIZE, seed=SEED):
    rng = np.random.RandomState(seed)
    val_idx = []
    for letter in "abcd":
        idx = np.where(df.answer.values == letter)[0]
        rng.shuffle(idx)
        val_idx.extend(idx[: val_size // 4].tolist())
    val_idx = sorted(val_idx)
    mask = np.zeros(len(df), dtype=bool); mask[val_idx] = True
    return df[~mask].reset_index(drop=True), df[mask].reset_index(drop=True)

tr_df, val_df = make_split(train_df)
split_md5 = hashlib.md5(",".join(val_df.id.tolist()).encode()).hexdigest()
print(f"train {len(tr_df):,} / valid {len(val_df):,} | split md5 = {split_md5}")
print("valid 정답 분포:", val_df.answer.value_counts().sort_index().to_dict())
assert split_md5 == EXPECTED_SPLIT_MD5, "⚠️ 검증 분할이 기준과 다릅니다 → 기존 멤버 예측과 비교할 수 없습니다"
print("✅ 팀 공통 분할과 일치")

## 6. 프롬프트 (학습 때와 동일)

In [ ]:
PROMPT_VERSION = "baseline_v1"
SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")
LETTERS = "abcd"

def build_mc_prompt(question, options):
    a, b, c, d = options
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

def build_messages(row, perm=(0, 1, 2, 3)):
    # perm[k] = 화면의 k번째 자리에 놓을 '원래 선지'의 인덱스
    opts = [str(row[LETTERS[j]]) for j in perm]
    user_text = build_mc_prompt(str(row["question"]), opts)
    if globals().get("USE_SYSTEM_ROLE", True):
        return [{"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    # system 역할이 없는 템플릿(일부 Gemma 등): 지시문을 user 메시지 앞에 붙임
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": SYSTEM_INSTRUCT + "\n\n" + user_text}]}]

def load_image(rel):
    im = Image.open(img_path(rel))
    im = ImageOps.exif_transpose(im)      # 촬영 방향(EXIF) 보정 — 없으면 그대로
    return im.convert("RGB")

## 7. 기준 앙상블 → 애매한 문항 고르기 (GPU 불필요, E9와 같은 문항)

In [ ]:
# ── 기준 앙상블 3개를 멤버 예측 파일로 다시 만들고, 애매한 문항(1·2위 확률 차 < TAU_GATE)을 고릅니다 (GPU 불필요) ──
_CACHE = {}
def _probs(d): return d[["p_a", "p_b", "p_c", "p_d"]].to_numpy(dtype=float)
def _load_preds(path, ids):
    d = pd.read_csv(path).drop_duplicates("id", keep="last").set_index("id")
    miss = [i for i in ids if i not in d.index]
    assert not miss, f"{os.path.basename(path)}에 없는 문항 {len(miss)}개 (예: {miss[:3]})"
    return d.loc[ids]
def _read_log(run):
    p = os.path.join(OUT_DIR, run, "train_log.json")
    try: return json.load(open(p, encoding="utf-8")) if os.path.exists(p) else {}
    except Exception: return {}
def member_file(run, preset, split):
    d = os.path.join(OUT_DIR, run)
    if not os.path.isdir(d): return None, "폴더 없음"
    cv, ct = {}, {}
    for f in os.listdir(d):
        m = re.match(rf"^preds_final_ep(\d+)__{re.escape(preset)}__1000\.csv$", f)
        if m: cv[int(m.group(1))] = f; continue
        m = re.match(rf"^preds_final_ep(\d+)_test__{re.escape(preset)}__(\d+)\.csv$", f)
        if m and int(m.group(2)) == len(test_df): ct[int(m.group(1))] = f
    base_ = cv or ct
    if not base_: return None, "예측 파일 없음"
    ea = {int(k): v for k, v in (_read_log(run).get("epoch_acc") or {}).items()}
    ep = max(ea, key=lambda e: (ea[e], e)) if ea else None
    ep = ep if ep in base_ else max(base_)
    pool = cv if split == "val" else ct
    return (os.path.join(d, pool[ep]), f"ep{ep}") if ep in pool else (None, f"ep{ep} {split} 파일 없음")
def load(run, preset, split):
    key = (run, preset, split)
    if key not in _CACHE:
        f, why = member_file(run, preset, split)
        if f is None: raise FileNotFoundError(f"{run}@{preset} {split}: {why}")
        _CACHE[key] = _probs(_load_preds(f, VAL_IDS if split == "val" else TEST_IDS))
    return _CACHE[key]
T70 = np.exp(np.linspace(np.log(0.25), np.log(8.0), 70))          # Stage 3와 같은 온도 격자
def temp_scale(p, T):
    q = np.clip(p, 1e-12, 1.0) ** (1.0 / T); return q / q.sum(1, keepdims=True)
def fit_T(p, y):
    i = np.arange(len(y)); return float(T70[int(np.argmin([-np.mean(np.log(temp_scale(p, T)[i, y])) for T in T70]))])
def ens_probs(r, split, over=None):
    # over: {멤버: (검증 확률, test 확률)} → 그 멤버 확률을 바꿔서 같은 레시피로 앙상블 (온도는 바뀐 검증 확률로 다시 맞춤)
    w = np.asarray(r["weights"], float); w = w / w.sum(); out = 0.0
    for wi, (run, pre) in zip(w, r["members"]):
        Pv, Pt = over[run] if (over and run in over) else (load(run, pre, "val"), load(run, pre, "test"))
        P = Pv if split == "val" else Pt
        if r["cal"]: P = temp_scale(P, fit_T(Pv, Y_VAL))
        out = out + wi * P
    return out
def vote_of(arrs):
    A = np.stack(arrs, 1); cnt = np.stack([(A == k).sum(1) for k in range(4)], 1)
    return cnt.argmax(1)             # 동률이면 알파벳 앞 글자 = pandas mode(axis=1)[0]
def file_answers(fname):
    p = os.path.join(OUT_DIR, ENSEMBLE_SUBDIR, fname)
    if not os.path.exists(p): return None
    s = pd.read_csv(p).drop_duplicates("id", keep="last").set_index("id").reindex(TEST_IDS)["answer"]
    s = s.astype(str).str.strip().str.lower().map({L: i for i, L in enumerate(LETTERS)})
    return None if s.isna().any() else s.to_numpy().astype(int)
def margin(P):
    s = np.sort(P, axis=1); return s[:, -1] - s[:, -2]
def paired(a_old, a_new, y):
    c0, c1 = a_old == y, a_new == y
    f, b = int((~c0 & c1).sum()), int((c0 & ~c1).sum()); return f, b, f - b, int((a_old != a_new).sum())
def score_of(count): return math.floor(count / N_PUBLIC * 1e5) / 1e5
def count_of(score): return int(round(score * N_PUBLIC))

VAL_IDS, TEST_IDS = val_df.id.tolist(), test_df.id.tolist()
Y_VAL = val_df.answer.map(LETTERS.index).to_numpy()
BASE = {name: {s: ens_probs(r, s) for s in ("val", "test")} for name, r in RECIPES.items()}
A0 = {name: {s: BASE[name][s].argmax(1) for s in ("val", "test")} for name in RECIPES}
A0[VOTE["name"]] = {s: vote_of([A0[v][s] for v in VOTE["voters"]]) for s in ("val", "test")}
GATE_V = np.any([margin(BASE[n]["val"]) < TAU_GATE for n in RECIPES], axis=0)
GATE_T = np.any([margin(BASE[n]["test"]) < TAU_GATE for n in RECIPES], axis=0)
VAL_G, TEST_G = val_df[GATE_V].reset_index(drop=True), test_df[GATE_T].reset_index(drop=True)
_fa = file_answers(RECIPES["clean4_qg_hcx"]["file"])
print(f"기준 clean4 검증 {(A0['clean4_qg_hcx']['val'] == Y_VAL).mean() * 100:.2f}% | 다수결(레시피) 검증 {(A0[VOTE['name']]['val'] == Y_VAL).mean() * 100:.2f}%"
      + ("" if _fa is None else f" | clean4 test 재현 {int((_fa == A0['clean4_qg_hcx']['test']).sum()):,}/{len(TEST_IDS):,}"))
print(f"채점 대상(세 앙상블 중 하나라도 1·2위 확률 차 < {TAU_GATE}): 검증 {len(VAL_G)}문항 (그중 clean4 오답 {int((GATE_V & (A0['clean4_qg_hcx']['val'] != Y_VAL)).sum())}개 / 전체 오답 {int((A0['clean4_qg_hcx']['val'] != Y_VAL).sum())}개) | test {len(TEST_G)}문항")
print(f"→ 멤버 {len(MEMBERS_HR)}개 × (검증 {len(VAL_G)} × 해상도 {1 + len(HR_PRESETS)} + test {len(TEST_G) if RUN_TEST else 0} × 해상도 {len(HR_PRESETS)})  (up2.0 검증은 기존 예측 재현 확인용)")

## 8. 채점 함수 (stage2와 동일, 증강만 제거)

In [ ]:
IDENTITY = (0, 1, 2, 3)
TTA_PERMS = [(0, 1, 2, 3), (1, 2, 3, 0), (2, 3, 0, 1), (3, 0, 1, 2)]

def set_resolution(preset):
    global RES_PRESET, PIXEL_KW
    RES_PRESET = preset
    if FAMILY == "gemma4":
        PIXEL_KW = {"max_soft_tokens": int(RES_PRESETS_GEMMA4[preset])}
    else:
        mn, mx = RES_PRESETS[preset]
        PIXEL_KW = {"min_pixels": int(mn), "max_pixels": int(mx)}
# (해상도는 멤버 모델을 올린 뒤 set_resolution(PRESET)으로 정함)

def render(row, perm=IDENTITY):
    return processor.apply_chat_template(build_messages(row, perm), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW)

def encode_batch(rows, perms):
    texts  = [render(r, p) for r, p in zip(rows, perms)]
    images = [load_image(r["path"]) for r in rows]
    if FAMILY == "gemma4": images = [[im] for im in images]          # Gemma 프로세서는 샘플별 이미지 리스트를 받음
    return processor(text=texts, images=images, padding=True, return_tensors="pt", **PIXEL_KW)

def _last_logits(enc):
    enc = enc.to(model.device)
    with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        try:    out = model(**enc, logits_to_keep=1, use_cache=False)
        except TypeError: out = model(**enc)
    return out.logits[:, -1, :].float()

def unpermute(prob_slot, perms):
    probs = np.zeros_like(prob_slot)
    for i, p in enumerate(perms):
        for k, j in enumerate(p): probs[i, j] = prob_slot[i, k]
    return probs

@torch.inference_mode()
def score_rows(rows, perms=None):
    global BS
    perms = perms or [IDENTITY] * len(rows)
    enc = logits = None; oom = False
    try:
        enc = encode_batch(rows, perms)
        n_img_tok = (enc["input_ids"] == IMAGE_TOKEN_ID).sum(dim=1).tolist()
        logits = _last_logits(enc)
        if not torch.isfinite(logits).all(): raise FloatingPointError("로짓에 NaN/Inf 발생")
    except torch.cuda.OutOfMemoryError:
        oom = True
    if oom:
        del enc, logits; torch.cuda.empty_cache()
        if len(rows) == 1: raise RuntimeError("배치 1에서도 GPU 메모리 부족")
        h = len(rows) // 2; BS = max(1, min(BS, h)); print(f"  ⚠️ OOM → 평가 배치를 {BS}(으)로 줄입니다")
        p1, m1, t1 = score_rows(rows[:h], perms[:h]); p2, m2, t2 = score_rows(rows[h:], perms[h:])
        return np.concatenate([p1, p2]), np.concatenate([m1, m2]), t1 + t2
    logp = torch.log_softmax(logits, dim=-1)
    slot = torch.stack([torch.logsumexp(logp[:, ids], dim=1) for ids in LETTER_IDS], dim=1)
    mass = slot.exp().sum(dim=1).cpu().numpy()
    return unpermute(torch.softmax(slot, dim=1).cpu().numpy().astype(np.float64), perms), mass, n_img_tok

def score_rows_tta(rows, tta):
    ps, ms, nt = [], [], None
    for perm in (TTA_PERMS if tta else [IDENTITY]):
        p, m, nt = score_rows(rows, [perm] * len(rows)); ps.append(p); ms.append(m)
    return np.mean(ps, axis=0), np.mean(ms, axis=0), nt

def predict_df(df, tag, tta=False, resume=True, verbose=True, save_every=25):
    # df 전체를 채점. resume=True면 CSV에 이어 쓰며, 같은 tag로 다시 부르면 남은 문항만 처리
    was_training = model.training; model.eval()
    out_path = os.path.join(RUN_DIR, f"preds_{tag}__{RES_PRESET}{'__tta4' if tta else ''}__{len(df)}.csv")
    done_ids = set(pd.read_csv(out_path).id) if (resume and os.path.exists(out_path)) else set()
    if not resume and os.path.exists(out_path): os.remove(out_path)
    todo = df[~df.id.isin(done_ids)]
    if SKIP_MISSING_IMAGES: todo = todo[todo.path.map(lambda p: os.path.exists(img_path(p)))]
    rows = todo.to_dict("records"); recs, t0, n_done, nb = [], time.time(), 0, 0
    if verbose: print(f"[{os.path.basename(out_path)}] 전체 {len(df):,} | 완료 {len(done_ids):,} | 남음 {len(rows):,}")
    while n_done < len(rows):
        batch = rows[n_done: n_done + BS]
        probs, mass, ntok = score_rows_tta(batch, tta)
        for r, p, m, t in zip(batch, probs, mass, ntok):
            recs.append({"id": r["id"], "pred": LETTERS[int(p.argmax())], "p_a": p[0], "p_b": p[1], "p_c": p[2], "p_d": p[3],
                         "letter_mass": float(m), "img_tokens": int(t)})
        n_done += len(batch); nb += 1
        if nb % save_every == 0 or n_done >= len(rows):
            pd.DataFrame(recs).to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False); recs = []
            if verbose:
                el = time.time() - t0
                print(f"  {n_done:>5}/{len(rows)} | {el / n_done:.2f}s/문항 | 남은 시간 ~{(len(rows) - n_done) * el / n_done / 60:.0f}분", flush=True)
    if was_training: model.train()
    return pd.read_csv(out_path).drop_duplicates("id", keep="last"), out_path

CATS = {"전화번호": r"전화|번호|연락처", "가격/금액": r"가격|얼마|금액|요금|할인", "이름/상호/제목": r"이름|상호|제목|브랜드|가게|매장|간판|업체",
        "날짜/시간": r"날짜|언제|시간|기간|요일|몇 시|년도|연도", "위치/방향/층": r"어디|위치|방향|몇 층|층에|왼쪽|오른쪽|위쪽|아래",
        "개수/숫자": r"몇 개|몇 명|몇 번|개수|몇 가지|숫자", "색상": r"색|컬러", "부정형": r"아닌|않는|없는|틀린|잘못"}

def evaluate(df, tag, tta=False, resume=True, verbose=True, by_type=True):
    pred, _ = predict_df(df, tag, tta=tta, resume=resume, verbose=verbose)
    m = df.merge(pred, on="id", how="inner"); m["correct"] = (m.pred == m.answer)
    acc = float(m.correct.mean()); n = len(m); se = math.sqrt(max(acc * (1 - acc), 1e-9) / n)
    print(f"▶ {tag} | {RES_PRESET}{' +TTA4' if tta else ''} | 정확도 {acc * 100:.2f}% (n={n}, ±{1.96 * se * 100:.1f}%p) | letter_mass {m.letter_mass.mean():.3f}")
    if by_type:
        for k, pat in CATS.items():
            sel = m.question.str.contains(pat)
            if sel.sum() >= 10: print(f"    {k:<12} {m.correct[sel].mean() * 100:5.1f}%  (n={int(sel.sum())})")
    lp = os.path.join(OUT_DIR, "results_log_s2.csv")
    log = {"time": datetime.datetime.now().strftime("%m-%d %H:%M"), "run": f"{RUN_NAME}/{tag}", "model": str(MODEL_ID),
           "res": RES_PRESET, "tta": int(tta), "n": n, "acc": round(acc * 100, 2), "ci95": round(1.96 * se * 100, 2),
           "letter_mass": round(float(m.letter_mass.mean()), 3), "gpu": GPU_NAME, "split_md5": split_md5[:8]}
    pd.DataFrame([log]).to_csv(lp, mode="a", header=not os.path.exists(lp), index=False)
    return acc, m

## 9. 멤버 로드 함수 (bf16 베이스 + 기존 어댑터, merge 없음)

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel, load_peft_weights, get_peft_model_state_dict
import transformers, peft, gc, contextlib
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다 (런타임 → 런타임 유형 변경)"
DEVICE = "cuda"
GPU_NAME = torch.cuda.get_device_name(0); _cap = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
assert _cap[0] >= 8, f"{GPU_NAME}은 bf16을 지원하지 않습니다 → G4 또는 A100 80GB를 쓰세요"
DTYPE = torch.bfloat16
BS = (8 if VRAM_GB >= 38 else 4 if VRAM_GB >= 20 else 2) if BATCH_SIZE == "auto" else int(BATCH_SIZE)

def _free():
    for _v in ("model", "base", "enc", "logits", "out"):
        if _v in globals(): del globals()[_v]
    gc.collect(); torch.cuda.empty_cache()

def _drop_hf_cache(model_id):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
        d = os.path.join(HF_HUB_CACHE, "models--" + model_id.replace("/", "--"))
        if os.path.isdir(d): shutil.rmtree(d, ignore_errors=True); print(f"  디스크 확보: {model_id} 캐시 삭제")
    except Exception as e: print("  (캐시 삭제 생략:", e, ")")

def _norm_key(k):
    k = k.replace("base_model.model.", "", 1); i = k.rfind("language_model.")
    return k[i + len("language_model."):] if i >= 0 else k
def check_adapter_weights(model, adapter_dir):
    # E1과 같은 점검: 어댑터 파일의 LoRA 텐서가 전부 값 그대로 모델에 들어갔는지 (PEFT는 이름이 안 맞는 키를 조용히 건너뛸 수 있음)
    fw = load_peft_weights(adapter_dir, device="cpu"); mw = {k: v.detach() for k, v in get_peft_model_state_dict(model).items()}
    if set(fw) != set(mw): fw = {_norm_key(k): v for k, v in fw.items()}; mw = {_norm_key(k): v for k, v in mw.items()}
    assert set(fw) == set(mw), f"어댑터 파일과 모델의 LoRA 키가 다릅니다 (파일에만 {len(set(fw) - set(mw))}개, 모델에만 {len(set(mw) - set(fw))}개)"
    worst = max((v.to(mw[k].dtype).float() - mw[k].float().cpu()).abs().max().item() for k, v in fw.items())
    assert worst == 0.0, f"어댑터 파일과 모델의 LoRA 값이 다릅니다 (최대 차이 {worst:.3g})"
    assert any("lora_B" in k and v.abs().max().item() > 0 for k, v in fw.items()), "lora_B가 전부 0입니다 → 학습되지 않은 어댑터"
    return len(fw)

def load_member(run):
    # 멤버 하나를 bf16 베이스 + 기존 어댑터(merge 없음)로 올리고, 채점 함수가 쓰는 전역값을 그 멤버에 맞춤
    global processor, tok, FAMILY, IMAGE_TOKEN_ID, TEMPLATE_KW, USE_SYSTEM_ROLE, base, model, LETTER_IDS, MODEL_ID
    inf = MEMBER_INFO[run]; MODEL_ID = inf["model_id"]
    _sz = re.search(r"(?<![\d.])(\d+)B", MODEL_ID, flags=re.I)
    need = int(_sz.group(1)) * 2.05 + 6 if _sz else 0.0
    if VRAM_GB < need: raise RuntimeError(f"bf16 {MODEL_ID}에는 약 {need:.0f}GB가 필요한데 {GPU_NAME}은 {VRAM_GB:.0f}GB입니다 → G4 또는 A100 80GB")
    processor = AutoProcessor.from_pretrained(MODEL_ID); processor.tokenizer.padding_side = "left"; tok = processor.tokenizer
    FAMILY = "gemma4" if type(processor).__name__.startswith("Gemma4") else "qwen"
    IMAGE_TOKEN_ID = getattr(processor, "image_token_id", None)
    if IMAGE_TOKEN_ID is None: IMAGE_TOKEN_ID = tok.convert_tokens_to_ids(getattr(processor, "image_token", "<|image_pad|>"))
    _tmpl = processor.chat_template if isinstance(processor.chat_template, str) else json.dumps(processor.chat_template)
    TEMPLATE_KW = {"enable_thinking": False} if "enable_thinking" in (_tmpl or "") else {}
    try:
        processor.apply_chat_template(build_messages({"question": "q", "a": "1", "b": "2", "c": "3", "d": "4"}), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW); USE_SYSTEM_ROLE = True
    except Exception:
        USE_SYSTEM_ROLE = False
    t0 = time.time()
    base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=DTYPE, attn_implementation="sdpa", device_map={"": 0})
    base.config.use_cache = False
    assert not [n for n, m in base.named_modules() if any(k in type(m).__name__ for k in ("Linear4bit", "Linear8bit"))], "양자화된 층이 있습니다 (bf16이어야 함)"
    model = PeftModel.from_pretrained(base, inf["adapter_dir"], is_trainable=False).eval()
    n_lora = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A"))
    n_vis = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A") and any(k in n for k in ("visual", "vision_tower", "vision_model")))
    assert n_lora > 0 and n_vis == 0, f"LoRA 층 {n_lora}개, 비전 타워 {n_vis}개 → 어댑터 부착 이상"
    nk = check_adapter_weights(model, inf["adapter_dir"])
    if "qwen3_5" in str(getattr(base.config, "model_type", "")):
        import importlib.util
        if importlib.util.find_spec("fla") is None: print("  ⚠️⚠️ flash-linear-attention이 없습니다 → 2번 셀 실행 후 '세션 다시 시작'")
    LETTER_IDS = []
    for L in LETTERS:
        ids = []
        for s in (L, L.upper(), " " + L, " " + L.upper()):
            t = tok.encode(s, add_special_tokens=False)
            if len(t) == 1 and t[0] not in ids: ids.append(t[0])
        assert ids and len(tok.encode(L, add_special_tokens=False)) == 1, f"'{L}' 가 단일 토큰이 아닙니다"
        LETTER_IDS.append(ids)
    set_resolution(PRESET)
    print(f"  모델 로드 {time.time() - t0:.0f}s | {type(base).__name__} | 계열 {FAMILY} | LoRA 키 {nk}개 값까지 일치 | 템플릿 인자 {TEMPLATE_KW} | system 역할 {USE_SYSTEM_ROLE}")
print(f"GPU: {GPU_NAME} ({VRAM_GB:.0f}GB) | bf16 | 평가 배치 {BS}")

## 10. 멤버별 채점: 검증 up2.0 재현 확인 → 검증 up3.0·up4.0 → test up3.0·up4.0

In [ ]:
# ── 멤버마다: 모델 로드 → 사전 점검 → 애매한 문항을 해상도별로 채점 (검증 up2.0 재현 확인 → 검증 고해상도 → test 고해상도). 끊겨도 다시 실행하면 남은 것만 ──
RES_COLS = ["id", "preset", "p_a", "p_b", "p_c", "p_d", "letter_mass", "img_tokens"]
SPLIT_DF = {"val": VAL_G, "test": TEST_G}
def _res_path(run, split): return os.path.join(E5_DIR, run, f"res_{split}.csv")
def _done(run, split):
    p = _res_path(run, split)
    if not os.path.exists(p): return set()
    d = pd.read_csv(p); return set(zip(d.id, d.preset))
def _plan():
    return [("val", "up2.0")] + [("val", pr) for pr in HR_PRESETS] + ([("test", pr) for pr in HR_PRESETS] if RUN_TEST else [])
def _skipped(run): return set() if RETRY_SKIPPED else set((E5_INFO.get("infeasible") or {}).get(run, []))
def _remaining(run):
    dn = {s: _done(run, s) for s in ("val", "test")}
    return {(s, pr): 0 if pr in _skipped(run) else sum((i, pr) not in dn[s] for i in SPLIT_DF[s].id) for s, pr in _plan()}
def score_res(run, split, preset, save_every=10):
    global BS
    path = _res_path(run, split); os.makedirs(os.path.dirname(path), exist_ok=True)
    done = _done(run, split); rows = [r for r in SPLIT_DF[split].to_dict("records") if (r["id"], preset) not in done]
    set_resolution(preset); BS = min(HR_BS.get(preset, 8), (8 if VRAM_GB >= 38 else 4 if VRAM_GB >= 20 else 2))
    print(f"  [{split} | {preset}] {len(rows):,}문항 남음 (배치 {BS})", flush=True)
    recs, n, nb, t0 = [], 0, 0, time.time()
    try:
        while n < len(rows):
            batch = rows[n: n + BS]
            p, m, nt = score_rows(batch)
            if n == 0 and preset != PRESET:
                # 해상도가 실제로 바뀌었는지: 모든 사진이 up2.0에선 최대 픽셀 이하, 고해상도에선 최소 픽셀 이상이어야 하므로 토큰 수로 바로 확인됨
                _lo = 0.85 * RES_PRESETS[preset][0] / RES_PRESETS[PRESET][1] * UP2_TOK_MAX
                if min(nt) < _lo:
                    raise RuntimeError(f"{preset} 해상도가 적용되지 않은 것 같습니다: 이미지 토큰 {list(nt)} < 기대 최소 {_lo:.0f} "
                                       f"(up2.0 사전 점검 최대 {UP2_TOK_MAX}) → 위 출력을 확인하세요")
            for r, pp, mm, tt in zip(batch, p, m, nt):
                recs.append({"id": r["id"], "preset": preset, "p_a": pp[0], "p_b": pp[1], "p_c": pp[2], "p_d": pp[3], "letter_mass": float(mm), "img_tokens": int(tt)})
            n += len(batch); nb += 1
            if nb % save_every == 0 or n >= len(rows):
                pd.DataFrame(recs, columns=RES_COLS).to_csv(path, mode="a", header=not os.path.exists(path), index=False); recs = []
                el = time.time() - t0; print(f"    {n:>5}/{len(rows)} | {el / n:.2f}s/문항 | 이미지 토큰 약 {int(np.mean(nt)):,}개 | 남은 시간 ~{(len(rows) - n) * el / n / 60:.0f}분", flush=True)
    except RuntimeError as e:
        if "메모리 부족" not in str(e): raise
        if recs: pd.DataFrame(recs, columns=RES_COLS).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
        E5_INFO.setdefault("infeasible", {}).setdefault(run, [])
        if preset not in E5_INFO["infeasible"][run]: E5_INFO["infeasible"][run].append(preset)
        json.dump(E5_INFO, open(INFO_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
        torch.cuda.empty_cache(); print(f"  ⚠️ {preset}: 배치 1에서도 GPU 메모리 부족 → 이 멤버는 {preset}를 건너뜁니다")
    set_resolution(PRESET)

E5_INFO.setdefault("members", {})
for i, run in enumerate(MEMBERS_HR):
    rem = _remaining(run)
    print(f"\n▶ {run} ({i + 1}/{len(MEMBERS_HR)}) | 남은 채점: " + ", ".join(f"{s}·{pr} {n:,}" for (s, pr), n in rem.items()))
    if sum(rem.values()) == 0: print("  이미 끝났습니다 → 건너뜀"); continue
    _free(); load_member(run)
    # 사전 점검: 어댑터 켬/끔 차이, letter_mass, 이미지 토큰 수가 기존 예측과 같은지 (up2.0)
    set_resolution(PRESET)
    _chk = VAL_G.head(4).to_dict("records") or val_df.head(4).to_dict("records")
    p_on, m_on, nt_on = score_rows(_chk)
    with model.disable_adapter(): p_off, _, _ = score_rows(_chk)
    _d = float(np.abs(p_on - p_off).max())
    _ref = pd.read_csv(MEMBER_INFO[run]["ref_val"]).drop_duplicates("id", keep="last").set_index("id").reindex([r["id"] for r in _chk])
    print(f"  사전 점검(up2.0): 어댑터 켬/끔 확률 최대 차이 {_d:.3f} | letter_mass {m_on.mean():.3f} | 이미지 토큰 {list(nt_on)}"
          + (f" (기존 {[int(t) for t in _ref.img_tokens]})" if "img_tokens" in _ref.columns else ""))
    assert _d > 1e-3, "어댑터를 켜고 꺼도 출력이 같습니다 → 어댑터가 적용되지 않았습니다. 위 출력을 확인하세요"
    assert m_on.mean() > 0.9, "letter_mass가 낮습니다 → 프롬프트·토큰 처리 문제. 위 출력을 확인하세요"
    UP2_TOK_MAX = int(max(nt_on))
    if "img_tokens" in _ref.columns and not _ref.img_tokens.isna().any():
        assert [int(t) for t in _ref.img_tokens] == [int(t) for t in nt_on], "이미지 토큰 수가 기존 예측과 다릅니다 → 해상도·프로세서가 달라 비교할 수 없습니다"
    E5_INFO["members"][run] = {"md5": MEMBER_INFO[run]["md5"], "model_id": MODEL_ID, "adapter": MEMBER_INFO[run]["adapter_dir"], "precision": "bf16", "gpu": GPU_NAME,
                               "presets": {pr: list(RES_PRESETS[pr]) for pr in ["up2.0"] + HR_PRESETS}, "tau_gate": TAU_GATE}
    json.dump(E5_INFO, open(INFO_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
    for s, pr in _plan():
        if pr in _skipped(run): print(f"  [{s} | {pr}] 메모리 부족으로 건너뜀(이전 실행 기록)"); continue
        score_res(run, s, pr)
    nxt = MEMBER_INFO[MEMBERS_HR[i + 1]]["model_id"] if i + 1 < len(MEMBERS_HR) else None
    if DELETE_HF_CACHE and nxt != MODEL_ID:
        _free(); _drop_hf_cache(MODEL_ID)
print("\n채점 끝 → 11번 셀에서 판정")

## 11. 판정 → 제출 후보
멤버별로 `up2.0 재현(%)`이 85% 이상, 해상도별 letter_mass가 0.9 이상이어야 정상입니다. 해상도별 정답 수(애매한 검증 문항 기준)도 함께 봅니다.

In [ ]:
# ── 판정 (GPU 불필요): 애매한 문항에서만 Q35·Q38 확률을 여러 해상도 평균으로 바꿔 같은 레시피로 앙상블 → 같은 문항끼리 비교 ──
V = {"lines": [], "cands": []}
def say(s): print(s); V["lines"].append(s)
IDX = {"val": {i: n for n, i in enumerate(VAL_IDS)}, "test": {i: n for n, i in enumerate(TEST_IDS)}}
GIDS = {"val": VAL_G.id.tolist(), "test": TEST_G.id.tolist()}
gv = np.array([IDX["val"][i] for i in GIDS["val"]], int); gt = np.array([IDX["test"][i] for i in GIDS["test"]], int)
def res_mat(run, split, preset):
    # (애매한 문항 수, 4) — 그 해상도로 모든 문항이 채점됐을 때만, 아니면 (None, None, None)
    p = _res_path(run, split)
    if not os.path.exists(p): return None, None, None
    d = pd.read_csv(p); d = d[d.preset == preset].drop_duplicates("id", keep="last").set_index("id").reindex(GIDS[split])
    if d.p_a.isna().any(): return None, None, None
    return d[["p_a", "p_b", "p_c", "p_d"]].to_numpy(dtype=float), float(d.letter_mass.mean()), float(d.img_tokens.mean())
MEM_ROWS, OVER1, OVER_SINGLE, health, USED = [], {}, {p: {} for p in HR_PRESETS}, [], {}
for run in MEMBERS_HR:
    Pv0, Pt0 = load(run, PRESET, "val"), load(run, PRESET, "test")
    R2, m2, t2 = res_mat(run, "val", "up2.0")
    avail = []
    for pr in HR_PRESETS:
        if pr in _skipped(run): continue
        Rv, mv, tv = res_mat(run, "val", pr); Rt = res_mat(run, "test", pr)[0] if RUN_TEST else None
        if Rv is None or (RUN_TEST and Rt is None): continue
        avail.append((pr, Rv, Rt, mv, tv))
    row = {"멤버": run}
    if R2 is None or not avail:
        row["상태"] = "채점 미완료 → 원래 확률 사용"; MEM_ROWS.append(row); continue
    yg = Y_VAL[gv]; base_g = Pv0[gv]
    rep_ = float((R2.argmax(1) == base_g.argmax(1)).mean() * 100)
    row.update({"상태": "✅", "up2.0 재현(%)": round(rep_, 1), "애매한 검증 정답: up2.0(기존)": int((base_g.argmax(1) == yg).sum())})
    for pr, Rv, Rt, mv, tv in avail:
        row[f"{pr} 정답"] = int((Rv.argmax(1) == yg).sum()); row[f"{pr} letter_mass"] = round(mv, 3); row[f"{pr} 이미지 토큰"] = int(tv)
        if mv < 0.9: health.append(f"{run} {pr}: letter_mass {mv:.3f} < 0.9")
    row["해상도 평균 정답"] = int((np.mean([base_g] + [a[1] for a in avail], 0).argmax(1) == yg).sum())
    row["문항 수"] = len(gv)
    if rep_ < 85: health.append(f"{run}: up2.0 재현 {rep_:.1f}% < 85%")
    if m2 < 0.9: health.append(f"{run} up2.0: letter_mass {m2:.3f}")
    Pv1, Pt1 = Pv0.copy(), Pt0.copy()
    Pv1[gv] = np.mean([base_g] + [a[1] for a in avail], 0)
    if RUN_TEST: Pt1[gt] = np.mean([Pt0[gt]] + [a[2] for a in avail], 0)
    OVER1[run] = (Pv1, Pt1); USED[run] = [a[0] for a in avail]
    for pr, Rv, Rt, _, _ in avail:                                    # 참고용: 기존 대신 그 해상도만 쓴 경우
        Pv_, Pt_ = Pv0.copy(), Pt0.copy(); Pv_[gv] = Rv
        if RUN_TEST: Pt_[gt] = Rt
        OVER_SINGLE[pr][run] = (Pv_, Pt_)
    MEM_ROWS.append(row)
print(pd.DataFrame(MEM_ROWS).to_string(index=False))
COMPLETE = len(OVER1) == len(MEMBERS_HR) and all(set(USED[r]) == set(HR_PRESETS) - _skipped(r) for r in OVER1)
if not OVER1:
    say("채점이 끝난 멤버가 없습니다 → 10번 셀을 먼저 끝까지 실행하세요")
else:
    def build(over):
        P = {n: {s: ens_probs(r, s, over) for s in ("val", "test")} for n, r in RECIPES.items()}
        A = {n: {s: P[n][s].argmax(1) for s in ("val", "test")} for n in RECIPES}
        A[VOTE["name"]] = {s: vote_of([A[v][s] for v in VOTE["voters"]]) for s in ("val", "test")}
        return A
    A1 = build(OVER1)
    say(f"[H1: 해상도 평균] 반영 멤버: " + ", ".join(f"{r}(up2.0+{'+'.join(USED[r])})" for r in OVER1) + ("" if COMPLETE else " (미완료 있음)"))
    RES = {}
    for nm in ("clean4_qg_hcx", VOTE["name"]):
        f, b, n, D = paired(A0[nm]["val"], A1[nm]["val"], Y_VAL); RES[nm] = (n, D)
        say(f"  {nm}: 검증 {(A0[nm]['val'] == Y_VAL).mean() * 100:.2f}% → {(A1[nm]['val'] == Y_VAL).mean() * 100:.2f}% | 고침 {f} | 새로 틀림 {b} | 순이득 {n:+d} | 바뀐 답 {D}"
            + ("  (바뀐 답 10개 미만 → 검증만으로는 판단 안 함)" if D < 10 else ""))
    for pr in HR_PRESETS:                                              # 참고용 (판정에 쓰지 않음)
        if len(OVER_SINGLE[pr]) == len(OVER1):
            As = build(OVER_SINGLE[pr]); f, b, n, D = paired(A0["clean4_qg_hcx"]["val"], As["clean4_qg_hcx"]["val"], Y_VAL)
            say(f"  (참고) {pr}만 쓴 경우 clean4: 고침 {f} / 새로 틀림 {b} / 순이득 {n:+d} (바뀐 답 {D})")
    n1, D1 = RES["clean4_qg_hcx"]
    # 해상도를 올려 답이 바뀐 검증 문항을 이미지와 함께 저장 (작은 글씨를 실제로 더 잘 읽었는지 눈으로 확인)
    try:
        import base64, io, html as _h
        ch = np.where(A0["clean4_qg_hcx"]["val"] != A1["clean4_qg_hcx"]["val"])[0]
        parts = [f"<h2>E5 — 해상도 평균으로 clean4 답이 바뀐 검증 문항 {len(ch)}개</h2>"]
        for i in ch:
            r = val_df.iloc[i]; im = load_image(r.path); im.thumbnail((1000, 1000)); buf = io.BytesIO(); im.save(buf, "JPEG", quality=80)
            res = "고침" if A1["clean4_qg_hcx"]["val"][i] == Y_VAL[i] else ("새로 틀림" if A0["clean4_qg_hcx"]["val"][i] == Y_VAL[i] else "둘 다 틀림")
            opts = "".join(f"<li>({k}) {_h.escape(str(r[k]))}{' <b>← 정답</b>' if k == r.answer else ''}{' ← 기존' if LETTERS[A0['clean4_qg_hcx']['val'][i]] == k else ''}{' ← 해상도 평균' if LETTERS[A1['clean4_qg_hcx']['val'][i]] == k else ''}</li>" for k in LETTERS)
            parts.append(f"<div style='margin:18px 0;border-top:1px solid #ccc;padding-top:12px'><b>{r.id} — {res}</b><p>{_h.escape(str(r.question))}</p><ul>{opts}</ul>"
                         f"<img src='data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}'></div>")
        _hp = os.path.join(E5_DIR, "e5_changed_val.html")
        open(_hp, "w", encoding="utf-8").write("<meta charset='utf-8'><body style='font-family:sans-serif;max-width:1100px;margin:auto'>" + "".join(parts))
        print("답이 바뀐 검증 문항 리포트:", _hp)
    except Exception as e:
        print("(리포트 생략:", e, ")")
    if health:
        say("판정: 확인 필요 — " + "; ".join(health) + " → 위 출력을 확인하세요")
    elif not COMPLETE:
        say("판정: 대기 — 모든 멤버의 채점이 끝나면 다시 실행")
    elif n1 <= -3:
        say(f"판정: 불채택 — H1 검증 순이득 {n1:+d} (≤ −3). 공개로 확인할 필요 없음")
    elif RUN_TEST:
        say(f"판정: 공개 확인 — 검증 순이득 {n1:+d}(바뀐 답 {D1}). 아래 clean4 후보의 공개 점수로 결정합니다")
        def _write(name, base_name, A_new, A_ref):
            fa = file_answers(RECIPES[base_name]["file"] if base_name in RECIPES else VOTE["file"])
            base_ans = fa if fa is not None else A_ref
            chg = A_new != A_ref; cand = np.where(chg, A_new, base_ans)
            path = os.path.join(E5_DIR, f"submission_{name}.csv"); sub_df[["id"]].assign(answer=[LETTERS[i] for i in cand]).to_csv(path, index=False)
            return os.path.basename(path), int((cand != base_ans).sum())
        need = max(0, NET_MIN - n1)
        for nm in ("clean4_qg_hcx", VOTE["name"]):
            fn, Dn = _write(f"{nm}__e5_hires", nm, A1[nm]["test"], A0[nm]["test"]); b0 = count_of(PUBLIC[nm]); V["cands"].append((fn, Dn))
            line = f"  제출 후보: {fn} | 기존 {nm}({PUBLIC[nm]:.5f} = {b0:,}문항)과 다른 답 {Dn}개 (공개에 약 {Dn // 2}개)"
            if nm == "clean4_qg_hcx":
                line += f" | ◀ 판정용: 공개 {score_of(b0 + need):.5f}({b0 + need:,}문항) 이상이면 채택 (검증 {n1:+d} + 공개 ≥ +{NET_MIN}, 공개 ≥ 0)"
            say(line)
        say("  제출 순서: clean4 후보(판정용) → 다수결 후보(실제로 쓸 것). 판정은 clean4 후보 한 줄로만 합니다")
    else:
        say(f"판정: 검증만 확인 — 검증 순이득 {n1:+d}(바뀐 답 {D1}). 제출 후보를 만들려면 1번 셀 RUN_TEST=True로 10번 셀부터 다시 실행")
    E5_INFO.update({"verdict_lines": V["lines"], "cands": V["cands"]})
    json.dump(E5_INFO, open(INFO_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)

## 12. 결과 요약

In [ ]:
print(f"E5 | 애매한 문항 고해상도 | 기준 {PRESET} + {HR_PRESETS} | 채점 대상 검증 {len(VAL_G)} / test {len(TEST_G)} | GPU {globals().get('GPU_NAME', '-')}")
if E5_INFO.get("infeasible"): print("  메모리 부족으로 건너뛴 해상도:", E5_INFO["infeasible"])
for r in MEM_ROWS: print("  ", r)
for l in V["lines"]: print(l)